# Amazon ML Challenge 2026 — Track 04: Candidate Blocking Evaluation & Diagnostics
**Objective:** Evaluate candidate generation quality against ground truth annotations.
- 1. Pair Completeness / Candidate Recall ($PC$)
- 2. Reduction Ratio ($RR$) & Pairs Quality / Precision ($PQ$)
- 3. Candidate Recall across validation subsets
- 4. Inspection of labeled candidate pairs (True Positives vs True Negatives)
- 5. Root-cause diagnostic error analysis on missed true matches (False Negatives)


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import src

print("Evaluation pipeline initialized!")


## 1. Ground Truth Alignment & Split
We load ground truth annotations, map target IDs, and split into train and validation query sets.


In [ ]:
# Load aligned evaluation subset
SAMPLE_SIZE = 15000
s1_eval = src.load_source("train", 1, nrows=SAMPLE_SIZE)
s2_eval = src.load_source("train", 2, nrows=80000)
s3_eval = src.load_source("train", 3, nrows=80000)
targets_eval = pd.concat([s2_eval, s3_eval], ignore_index=True)

gt_eval = src.load_ground_truth(nrows=SAMPLE_SIZE)
gt_map = src.parse_ground_truth_map(gt_eval)

s1_lookup = s1_eval.set_index("entity_id").to_dict("index")
target_lookup = targets_eval.set_index("entity_id").to_dict("index")

print(f"S1 Queries: {len(s1_eval):,} | Targets Pool: {len(targets_eval):,} | Ground Truth Mappings: {len(gt_map):,}")


## 2. Candidate Generation & Recall Evaluation


In [ ]:
# Fit blocker and retrieve candidates
blocker = src.TokenInvertedIndexBlocker(max_token_freq=800, min_token_len=3, top_k=30)
blocker.fit(targets_eval)
candidates_eval = blocker.generate_candidates(s1_eval)

# Compute full blocking metrics
metrics = src.compute_blocking_metrics(
    candidate_pairs_df=candidates_eval,
    ground_truth_map=gt_map,
    total_query_count=len(s1_eval),
    total_target_count=len(targets_eval)
)

print("=== BLOCKING PERFORMANCE EVALUATION METRICS ===")
for k, v in metrics.items():
    print(f"{k:30s}: {v}")


## 3. Labeled Sample Inspection (True Positives vs False Candidates)
Inspect a balanced sample of candidate pairs with ground truth labels and feature attributes.


In [ ]:
labeled_samples = src.sample_candidate_pairs_with_labels(
    candidate_pairs_df=candidates_eval,
    ground_truth_map=gt_map,
    s1_lookup=s1_lookup,
    target_lookup=target_lookup,
    n_positives=5,
    n_negatives=5
)

display(labeled_samples[["source1_id", "candidate_id", "ground_truth_label", "blocking_score", "s1_name", "candidate_name", "country"]])


## 4. Diagnostic Error Analysis: Missed True Matches (False Negatives)
To achieve winning performance in the challenge, we inspect matches missed by candidate generation.


In [ ]:
missed_df = src.analyze_missed_pairs(
    candidate_pairs_df=candidates_eval,
    ground_truth_map=gt_map,
    s1_lookup=s1_lookup,
    target_lookup=target_lookup,
    sample_size=10
)

print(f"Examining {len(missed_df)} sample false negative pairs (missed by blocking):")
display(missed_df[["source1_id", "target_id", "s1_name", "target_name", "s1_address", "target_address"]])


## 5. Candidate Recall vs Budget Trade-off Curve
Evaluate how Candidate Recall scales as a function of the top-$K$ candidate budget.


In [ ]:
top_k_values = [5, 10, 15, 20, 30, 40]
recalls = []

for k in top_k_values:
    # Filter candidates to top k per query
    cands_k = candidates_eval.groupby("source1_entity_id").head(k)
    m = src.compute_blocking_metrics(cands_k, gt_map, len(s1_eval), len(targets_eval))
    recalls.append(m["pair_completeness_recall"])

plt.figure(figsize=(8, 4))
plt.plot(top_k_values, recalls, marker="o", linewidth=2, color="#2563eb")
plt.title("Candidate Recall vs. Candidate Budget (Top-K per Query)")
plt.xlabel("Top-K Candidates per Query Entity")
plt.ylabel("Candidate Recall (Pair Completeness)")
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Export final evaluation metrics to outputs/
metrics_df = pd.DataFrame([metrics])
metrics_out = PROJECT_ROOT / "outputs" / "blocking_evaluation_metrics.csv"
metrics_df.to_csv(metrics_out, index=False)
print(f"Blocking evaluation metrics exported to: {metrics_out}")
